In [ ]:
# @title 1. Environment Setup & Dependencies

import os
import shutil
import sys
import subprocess
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

print("📦 Installing dependencies...")

# 1. Uninstall incompatible versions
!pip uninstall -y scikit-image imageio

# 2. Install fixed versions for Colab
!pip install -q imageio==2.33.0 scikit-image==0.22.0
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q smplx chumpy trimesh moviepy gradio gdown

# 3. System dependencies
!apt-get install -y ffmpeg

# 4. Clone Repository
if not os.path.exists("motion-diffusion-model"):
    print("📂 Cloning repository...")
    !git clone https://github.com/GuyTevet/motion-diffusion-model.git
    %cd motion-diffusion-model
else:
    %cd motion-diffusion-model

print("✅ Environment Installed.")

In [ ]:
# @title 2. Data Setup & Auto-Fix (Hugging Face Mirror)


import os
import shutil
import urllib.request

# 1. Download basic assets (SMPL/GloVe) - usually these work or are optional for basic generation
print("⬇️ Downloading SMPL & GloVe...")
!chmod +x prepare/download_smpl_files.sh
!chmod +x prepare/download_glove.sh
!bash prepare/download_smpl_files.sh > /dev/null 2>&1
!bash prepare/download_glove.sh > /dev/null 2>&1

# 2. Setup Dataset Directory
target_dir = "dataset/HumanML3D"
os.makedirs(target_dir, exist_ok=True)

# 3. FIX: Download Mean.npy and Std.npy from Hugging Face Mirror
# The original Google Drive link fails, so we fetch these 2KB files from a public mirror.
print("🔧 Fixing missing dataset statistics (Mean.npy, Std.npy)...")

mirror_base = "https://huggingface.co/OpenRobotLab/MotionMillion/resolve/main/mean_std"
files_to_fix = ["Mean.npy", "Std.npy"]

for file_name in files_to_fix:
    save_path = os.path.join(target_dir, file_name)
    url = f"{mirror_base}/{file_name}"

    if not os.path.exists(save_path):
        print(f"   ☁️ Downloading {file_name} from Hugging Face mirror...")
        try:
            # Use wget for reliability in Colab
            !wget -q -O {save_path} {url}
        except Exception as e:
            print(f"   ❌ Failed to download {file_name}: {e}")

# 4. Verify Data
mean_exists = os.path.exists(os.path.join(target_dir, "Mean.npy"))
std_exists = os.path.exists(os.path.join(target_dir, "Std.npy"))

if mean_exists and std_exists:
    print(f"\n✅ Data setup SUCCESS! Files are ready in {target_dir}")
else:
    print("\n❌ Error: Still missing Mean.npy or Std.npy.")
    print("Please manually upload them if this persists.")

In [ ]:
# @title 3. Model Download (Zip File)


import os
import zipfile
from google.colab import files

file_id = "1PE0PK8e5a5j-7-Xhs5YET5U5pGh0c821"
zip_filename = "humanml_trans_enc_512.zip"
target_dir = "save"
final_model_path = "save/humanml_trans_enc_512/model000200000.pt"

os.makedirs(target_dir, exist_ok=True)

def verify_model():
    return os.path.exists(final_model_path)

def unzip_file(zip_path, extract_to):
    print(f"📂 Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("✅ Unzip complete.")

if not verify_model():
    print("🔄 Attempting automatic download...")
    try:
        !gdown --id {file_id} -O {zip_filename}
        if os.path.exists(zip_filename) and os.path.getsize(zip_filename) > 100 * 1024 * 1024:
            unzip_file(zip_filename, target_dir)
        else:
            if os.path.exists(zip_filename): os.remove(zip_filename)
    except Exception:
        pass

if verify_model():
    print(f"🎉 Model is ready at: {final_model_path}")
else:
    print("\n" + "="*60)
    print("🛑 AUTOMATIC DOWNLOAD FAILED (Google Drive Quota)")
    print("="*60)
    print("1. Download ZIP here: https://drive.google.com/file/d/1PE0PK8e5a5j-7-Xhs5YET5U5pGh0c821/view?usp=sharing")
    print("2. Upload 'humanml_trans_enc_512.zip' using the button below.")

    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".zip"):
            unzip_file(filename, target_dir)

    if verify_model():
        print(f"🎉 Success! Model installed.")
    else:
        print("❌ Error: Model not found. Please check the zip file.")

In [ ]:
# @title 4. Patch Source Code (Smart Path Fix)
# @markdown 修改源代码以跳过“空数据集”报错。

import os

# 1. Detect correct path
possible_paths = [
    "data_loaders/humanml/data/dataset.py",  # If inside repo
    "motion-diffusion-model/data_loaders/humanml/data/dataset.py" # If outside repo
]

file_path = None
for p in possible_paths:
    if os.path.exists(p):
        file_path = p
        break

if file_path is None:
    print(f"❌ Error: Could not find dataset.py in current directory: {os.getcwd()}")
    print("Please ensure you ran Step 1.")
else:
    print(f"🔧 Patching {file_path}...")

    try:
        with open(file_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        patched_count = 0

        for line in lines:
            # Comment out the specific assertion causing the crash
            if "assert len(self.t2m_dataset) > 1" in line and "#" not in line[:5]:
                print("   ✅ Found and disabled assertion: len(self.t2m_dataset) > 1")
                new_lines.append(f"        # {line.strip()}  # PATCHED BY COLAB\n")
                patched_count += 1
            else:
                new_lines.append(line)

        if patched_count > 0:
            with open(file_path, "w") as f:
                f.writelines(new_lines)
            print("🎉 Patch applied successfully!")
        else:
            print("⚠️ Assertion line not found or already patched. (This is fine if you ran it before)")

    except Exception as e:
        print(f"❌ Error patching file: {e}")

# Double check Mean/Std exist
print("\n🔍 Verifying Statistical Data...")
if os.path.exists("dataset/HumanML3D/Mean.npy") and os.path.exists("dataset/HumanML3D/Std.npy"):
    print("   ✅ Mean.npy and Std.npy are present.")
else:
    print("   ❌ Mean.npy or Std.npy are MISSING. Please re-run the 'Fix Data' step.")

In [ ]:
# @title 5. Run Web Service


import gradio as gr
import glob
import subprocess
import sys
import os

# Suppress Audio/Display errors in Colab
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"

model_path = "save/humanml_trans_enc_512/model000200000.pt"

# Verify everything before running
if not os.path.exists(model_path):
    print("❌ Error: Model (model000200000.pt) is missing. Go back to the 'Model Download' step.")
elif not os.path.exists("dataset/HumanML3D/Mean.npy"):
    print("❌ Error: Mean.npy is missing. Run the 'Fix Corrupt Data Files' cell above.")
else:
    def generate_motion(text_prompt, motion_length, seed, repetition):
        # Cleanup
        !rm -rf save/humanml_trans_enc_512/samples_*

        print(f"🎬 Generating: '{text_prompt}'...")

        cmd = [
            sys.executable, "-m", "sample.generate",
            "--model_path", model_path,
            "--text_prompt", text_prompt,
            "--motion_length", str(motion_length),
            "--seed", str(int(seed)),
            "--num_repetitions", str(int(repetition)),
            "--device", "0"
        ]

        try:
            # Run inference
            subprocess.run(cmd, check=True, capture_output=True, text=True)

            # Find result
            search_path = "save/humanml_trans_enc_512/samples_*/sample*_rep*.mp4"
            files = glob.glob(search_path)

            if not files:
                return None, "Generation ran but no video found. (Check console for hidden errors)"

            return max(files, key=os.path.getctime), "Success!"

        except subprocess.CalledProcessError as e:
            return None, f"Error:\n{e.stderr}"
        except Exception as e:
            return None, str(e)

    iface = gr.Interface(
        fn=generate_motion,
        inputs=[
            gr.Textbox(label="Prompt", value="a person is dancing"),
            gr.Slider(1, 9, value=3, step=0.5, label="Duration (s)"),
            gr.Number(value=42, label="Seed"),
            gr.Slider(1, 3, value=1, step=1, label="Repetitions")
        ],
        outputs=[gr.Video(), gr.Textbox(label="Log")],
        title="MDM: Motion Diffusion Model",
        description="Text-to-Motion Generation."
    )

    iface.launch(share=True, debug=True)